
# ingest circuit csv file
- read file using spark dataframe reader api
- add metadata columns
 > source file,
 > ingestion timestamp
- wrrite to bronze delta table 

### read csv file using dataframe reader api

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema =StructType([

    StructField('circuitId',    StringType()),
    StructField('url',          StringType()),
    StructField('circuitName',  StringType()),
    StructField('lat',          DoubleType()),
    StructField('long',         DoubleType()),
    StructField('locality',     StringType()),
    StructField('country',      StringType())
])


In [0]:
%python
circuits_df = (
                spark.read.format('csv')
                .option('header', True)
                .option('mode', 'FAILFAST')
                .schema(circuits_schema)
                .load('/Volumes/formular1/landing/files/circuits.csv')
               
               )

In [0]:
display(circuits_df)


###add metadat columns

In [0]:
from pyspark.sql import functions as F
circuits_final_df =( 
            circuits_df
                .withColumn('ingestion_timestamp', F.current_timestamp())
                .withColumn('source_file', F.col('_metadata.file_path'))
)

In [0]:
display(circuits_final_df)


###write to bronze delta table


In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable('formular1.bronze.circuits')
)

In [0]:
%sql
select * from formular1.bronze.circuits;